# 03 — Autocorrelation and ARIMA


In [1]:
from spy_alpha_research.data import (
    load_config, project_root, ensure_spy, save_frame, load_frame,
    price_series, compute_returns, make_target, split_by_dates,
)
ROOT = project_root()
exp = load_config(ROOT / "configs" / "experiment.yaml")
models = load_config(ROOT / "configs" / "models.yaml")
bt = load_config(ROOT / "configs" / "backtest.yaml")
FIG = ROOT / exp["paths"]["figures_dir"]
FIG.mkdir(parents=True, exist_ok=True)
PROC = ROOT / exp["paths"]["processed_dir"]

from spy_alpha_research.classical import acf_pacf_values, fit_arima_grid, ljung_box_pvalue
from spy_alpha_research.reporting import plot_acf_pacf
ret = load_frame(PROC / "spy_returns.parquet").iloc[:, 0]
train_idx, _ = split_by_dates(
    ret.index, exp["dates"]["train_end"], exp["dates"]["test_start"], exp["dates"]["end"]
)
train_ret = ret.loc[train_idx].dropna()


In [2]:
acf_v, pacf_v = acf_pacf_values(train_ret, nlags=40)
plot_acf_pacf(acf_v, pacf_v, FIG / "acf_pacf.png")
orders = [tuple(o) for o in models["arima"]["order_grid"]]
arima = fit_arima_grid(train_ret, order_grid=orders, max_iter=models["arima"]["max_iter"])
print(arima["order"], arima["aic"])
ljung_box_pvalue(arima["resid"])


c:\Users\guray\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\guray\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\guray\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\guray\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been pro

(1, 0, 2) -9299.446485719007


5.98613919586574e-17